<a href="https://colab.research.google.com/github/utamiu1807/-utami-creditcardclustering/blob/main/ML_Foundations_Model_Evaluation_and_Hyperparameter_Tuning_In_Class_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

In [ ]:
import importlib.util
import itertools
import os
import subprocess
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [ ]:
def plot_decision_boundary(ax, model, X, y, title):
    """Plot smooth probability regions and the 0.5 decision boundary for 2D data."""
    x_min, x_max = X[:, 0].min() - 5, X[:, 0].max() + 5
    y_min, y_max = X[:, 1].min() - 5, X[:, 1].max() + 5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 350),
        np.linspace(y_min, y_max, 350),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = model.predict_proba(grid)[:, 1].reshape(xx.shape)

    ax.contourf(xx, yy, probs, levels=25, cmap="coolwarm", alpha=0.45)
    ax.contour(xx, yy, probs, levels=[0.5], colors="black", linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=20, edgecolor="k", linewidth=0.25)
    ax.set_title(title)
    ax.set_xlabel("MonthlyCharges")
    ax.set_ylabel("TenureMonths")


In [ ]:
import os
print(f"Number of processors/threads: {os.cpu_count()}")

In [ ]:
import psutil

ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
print(f"Used RAM: {ram.used / (1024**3):.2f} GB")

## Read Dataset

In [ ]:
df = pd.read_csv('https://www.dropbox.com/scl/fi/lle2wyziwywf8kxaq56na/telco_churn.csv?rlkey=kh9571prkkg8uki7mahnwunml&st=da4ou0gy&dl=1')

df.head()

## Data Exploration and Pre-processing

In [ ]:
# Drop unnecessary columns
df = df.drop(['Country', 'State', 'ChurnReason', 'ChurnLabel', "CustomerID"], axis=1)

In [ ]:
if df["TotalCharges"].isnull().sum() > 0:
    df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [ ]:
df.shape

In [ ]:
# Show all columns that have missing values
df.columns[df.isnull().any()]

In [ ]:
# # Starter predictor variables
selected_columns = [
    "TenureMonths",
    "Contract",
    "MonthlyCharges",
    "InternetService",
    "TechSupport",
    "ChurnValue"
]

df = df[selected_columns]

# Manual Train Test Split: Hyperparameter Tuning of Regularization

## Split Train and Test

In [ ]:
# Drop the target 'ChurnValue' from the features to prevent data leakage
X_model = pd.get_dummies(df.drop("ChurnValue", axis=1), drop_first=True)
y = df["ChurnValue"].astype(int)

In [ ]:
# We split indices so we can reuse the exact same rows in all sections.
all_idx = df.index.to_numpy()

idx_train, idx_temp = train_test_split(
    all_idx,
    test_size=0.35,
    stratify=y,
    random_state=42,
)

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,
    stratify=y.loc[idx_temp],
    random_state=42,
)

X_train = X_model.loc[idx_train]
X_val = X_model.loc[idx_val]
X_test = X_model.loc[idx_test]

y_train = y.loc[idx_train]
y_val = y.loc[idx_val]
y_test = y.loc[idx_test]

print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

## Manually Tune Hyperparameters

In [ ]:
# For visual decision boundaries, we use 2 numeric telecom features.
X_reg = df[["MonthlyCharges", "TenureMonths"]].copy()
X_reg_train = X_reg.loc[idx_train].values
X_reg_val = X_reg.loc[idx_val].values
y_reg_train = y.loc[idx_train].values
y_reg_val = y.loc[idx_val].values

# reg_configs = {
#     "Almost No Reg (L2, C=1e6)": dict(penalty="l2", C=1e6, solver="lbfgs", l1_ratio=None),
#     "Strong L2 (C=0.03)": dict(penalty="l2", C=0.03, solver="lbfgs", l1_ratio=None),
#     "L1 (C=0.08)": dict(penalty="l1", C=0.08, solver="saga", l1_ratio=None),
#     "Elastic Net (C=0.12, l1_ratio=0.5)": dict(
#         penalty="elasticnet", C=0.12, solver="saga", l1_ratio=0.5
#     ),
# }


reg_configs = {
    "No Reg (L2, C=1e6)": dict(penalty="l2", C=1e6, solver="lbfgs", l1_ratio=None),
    "Moderate L2 (C=1.0)": dict(penalty="l2", C=1.0, solver="lbfgs", l1_ratio=None),
    "Very Strong L2 (C=0.001)": dict(penalty="l2", C=0.001, solver="lbfgs", l1_ratio=None),
    "Strong L1 (C=0.01)": dict(penalty="l1", C=0.01, solver="saga", l1_ratio=None),
}

reg_rows = []
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for i, (name, cfg) in enumerate(reg_configs.items()):
    model = Pipeline(
        steps=[
            ("poly", PolynomialFeatures(degree=5, include_bias=False)),
            ("scale", StandardScaler()),
            (
                "logreg",
                LogisticRegression(
                    penalty=cfg["penalty"],
                    C=cfg["C"],
                    l1_ratio=cfg["l1_ratio"],
                    solver=cfg["solver"],
                    max_iter=6000,
                    random_state=42,
                ),
            ),
        ]
    )
    model.fit(X_reg_train, y_reg_train)

    train_acc = accuracy_score(y_reg_train, model.predict(X_reg_train))
    val_acc = accuracy_score(y_reg_val, model.predict(X_reg_val))
    val_auc = roc_auc_score(y_reg_val, model.predict_proba(X_reg_val)[:, 1])

    # Count how many polynomial coefficients are effectively non-zero.
    nonzero = int(np.sum(np.abs(model.named_steps["logreg"].coef_[0]) > 1e-6))

    reg_rows.append(
        {
            "model": name,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "val_roc_auc": val_auc,
            "nonzero_coefficients": nonzero,
        }
    )

    plot_decision_boundary(axes[i], model, X_reg.values, y.values, name)

In [ ]:
plt.suptitle("Regularization on Telecom Data (2D View)", y=1.02, fontsize=18)
plt.tight_layout()
plt.show()

reg_table = pd.DataFrame(reg_rows).sort_values("val_roc_auc", ascending=False)
print("\nRegularization comparison table:")
print(reg_table.to_string(index=False))

# Sweep C values for L1 and L2 to show performance + sparsity trends.
c_values = np.logspace(-3, 3, 16)
sweep_rows = []

for penalty, solver in [("l1", "saga"), ("l2", "lbfgs")]:
    for c_val in c_values:
        sweep_model = Pipeline(
            steps=[
                ("poly", PolynomialFeatures(degree=5, include_bias=False)),
                ("scale", StandardScaler()),
                (
                    "logreg",
                    LogisticRegression(
                        penalty=penalty,
                        C=c_val,
                        solver=solver,
                        max_iter=20000,
                        random_state=42,
                    ),
                ),
            ]
        )

        sweep_model.fit(X_reg_train, y_reg_train)
        val_probs = sweep_model.predict_proba(X_reg_val)[:, 1]
        nonzero = int(np.sum(np.abs(sweep_model.named_steps["logreg"].coef_[0]) > 1e-6))

        sweep_rows.append(
            {
                "penalty": penalty.upper(),
                "C": c_val,
                "val_roc_auc": roc_auc_score(y_reg_val, val_probs),
                "nonzero_coefficients": nonzero,
            }
        )

sweep_df = pd.DataFrame(sweep_rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.lineplot(data=sweep_df, x="C", y="val_roc_auc", hue="penalty", marker="o", ax=axes[0])
axes[0].set_xscale("log")
axes[0].set_title("Validation ROC AUC vs C")
axes[0].set_xlabel("C (larger C = weaker regularization)")
axes[0].set_ylabel("Validation ROC AUC")

sns.lineplot(
    data=sweep_df,
    x="C",
    y="nonzero_coefficients",
    hue="penalty",
    marker="o",
    ax=axes[1],
)
axes[1].set_xscale("log")
axes[1].set_title("Model Complexity vs C")
axes[1].set_xlabel("C (larger C = weaker regularization)")
axes[1].set_ylabel("Non-zero Coefficients")

plt.tight_layout()
plt.show()

# Manual Parameter Tuning

In [ ]:
# Define constants
SEED = 42
SEARCH_N_JOBS = -1

QUICK_MODE = False

if QUICK_MODE:
    manual_grid = {
        "n_estimators": [80, 160],
        "max_depth": [None, 8],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"],
    }
else:
    manual_grid = {
        "n_estimators": [80, 160, 300],
        "max_depth": [None, 8, 16],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
    }

manual_trials = list(
    itertools.product(
        manual_grid["n_estimators"],
        manual_grid["max_depth"],
        manual_grid["min_samples_leaf"],
        manual_grid["max_features"],
    )
)

manual_rows = []
manual_log_txt = Path("manual_search_log.txt")

with manual_log_txt.open("w", encoding="utf-8") as f:
    f.write("Manual Hyperparameter Search Log (Telco)\n")
    f.write("=" * 78 + "\n")

    for trial_idx, (n_est, depth, leaf, max_feat) in enumerate(manual_trials, start=1):
        model = RandomForestClassifier(
            n_estimators=n_est,
            max_depth=depth,
            min_samples_leaf=leaf,
            max_features=max_feat,
            random_state=SEED,
            n_jobs=SEARCH_N_JOBS,
        )
        model.fit(X_train, y_train)

        val_probs = model.predict_proba(X_val)[:, 1]
        val_pred = (val_probs >= 0.5).astype(int)

        val_acc = accuracy_score(y_val, val_pred)
        val_auc = roc_auc_score(y_val, val_probs)

        row = {
            "trial": trial_idx,
            "n_estimators": n_est,
            "max_depth": depth if depth is not None else "None",
            "min_samples_leaf": leaf,
            "max_features": max_feat,
            "val_accuracy": val_acc,
            "val_roc_auc": val_auc,
        }
        manual_rows.append(row)

        line = (
            f"Trial {trial_idx:03d} | n_estimators={n_est:>3} | max_depth={str(depth):>4} "
            f"| min_samples_leaf={leaf} | max_features={max_feat:>4} "
            f"| val_acc={val_acc:.4f} | val_auc={val_auc:.4f}"
        )
        print(line)
        f.write(line + "\n")

manual_df = pd.DataFrame(manual_rows).sort_values("val_accuracy", ascending=False)
manual_df.to_csv("manual_search_log.csv", index=False)

best_manual = manual_df.iloc[0]
best_manual_params = {
    "n_estimators": int(best_manual["n_estimators"]),
    "max_depth": None if best_manual["max_depth"] == "None" else int(best_manual["max_depth"]),
    "min_samples_leaf": int(best_manual["min_samples_leaf"]),
    "max_features": best_manual["max_features"],
}

print("\nTop 10 manual settings by validation val_auc:")
print(manual_df.head(10).to_string(index=False))
print("\nSaved logs: manual_search_log.txt and manual_search_log.csv")

# Heatmap view of manual search (slice for readability).
heatmap_df = (
    manual_df[(manual_df["min_samples_leaf"] == 1) & (manual_df["max_features"] == "sqrt")]
    .pivot_table(index="max_depth", columns="n_estimators", values="val_roc_auc", aggfunc="max")
)

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("Manual Search: Validation Accuracy Heatmap (leaf=1, max_features=sqrt)")
plt.xlabel("n_estimators")
plt.ylabel("max_depth")
plt.tight_layout()
plt.show()

# GridSearchCV

In [ ]:
print("GridSearchCV running...")

GRID_CV_FOLDS = 5

# Define constants here in case the manual search cell wasn't run
SEED = 42
SEARCH_N_JOBS = -1
QUICK_MODE = False

if QUICK_MODE:
    grid_params = {
        "n_estimators": [80, 160],
        "max_depth": [None, 8],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"],
    }
else:
    grid_params = {
        "n_estimators": [80, 160, 300],
        "max_depth": [None, 8, 16],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
    }

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=SEED, n_jobs=SEARCH_N_JOBS),
    param_grid=grid_params,
    scoring="accuracy",
    cv=2 if QUICK_MODE else GRID_CV_FOLDS,
    n_jobs=SEARCH_N_JOBS,
    verbose=1,
)
grid_search.fit(X_train, y_train)

grid_results = pd.DataFrame(grid_search.cv_results_).sort_values("mean_test_score", ascending=False)

print("\nBest GridSearchCV params:")
print(grid_search.best_params_)
print(f"Best GridSearchCV mean CV Accuracy: {grid_search.best_score_:.4f}")

grid_results.to_csv("grid_search_results.csv", index=False)

try:
    from google.colab import files
    files.download("grid_search_results.csv")
except ImportError:
    print("Google Colab files module not found. File saved locally as 'grid_search_results.csv'.")

print("\nAll Grid Search Results:")
grid_results

# RandomizedSearchCV

In [ ]:
print("RandomizedSearchCV running...")

RANDOM_CV_FOLDS = 5
RANDOM_N_ITER = 36

random_params = {
    "n_estimators": list(range(80, 501, 20)),
    "max_depth": [None] + list(range(4, 25)),
    "min_samples_split": [2, 4, 6, 8, 10, 12],
    "min_samples_leaf": [1, 2, 3, 4, 5, 6],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
}

# Calculate and print total possible combinations
total_combinations = 1
for p in random_params.values():
    total_combinations *= len(p)
print(f"Total possible combinations for Random Search: {total_combinations}")

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=SEED, n_jobs=SEARCH_N_JOBS),
    param_distributions=random_params,
    n_iter=8 if QUICK_MODE else RANDOM_N_ITER,
    scoring="accuracy", # alternatively "roc_auc",
    cv=2 if QUICK_MODE else RANDOM_CV_FOLDS,
    n_jobs=SEARCH_N_JOBS,
    random_state=SEED,
    verbose=1,
)
random_search.fit(X_train, y_train)

random_results = pd.DataFrame(random_search.cv_results_).sort_values("mean_test_score", ascending=False)

print("\nBest RandomizedSearchCV params:")
print(random_search.best_params_)
print(f"Best RandomizedSearchCV mean CV Accuracy: {random_search.best_score_:.4f}")

random_results.to_csv("random_results.csv", index=False)

random_results

# Compare Across Tuning Approaches

In [ ]:
print("\n4D) Comparing best models from manual/grid/random...")

manual_best_model = RandomForestClassifier(**best_manual_params, random_state=SEED, n_jobs=SEARCH_N_JOBS)
grid_best_model = clone(grid_search.best_estimator_)
random_best_model = clone(random_search.best_estimator_)

candidate_models = {
    "Manual Search Best": manual_best_model,
    "Grid Search Best": grid_best_model,
    "Random Search Best": random_best_model,
}

candidate_rows = []
for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    # Evaluate on Test Set
    test_probs = model.predict_proba(X_test)[:, 1]
    test_pred = (test_probs >= 0.5).astype(int)
    candidate_rows.append(
        {
            "model": name,
            "test_accuracy": accuracy_score(y_test, test_pred),
            "test_roc_auc": roc_auc_score(y_test, test_probs),
        }
    )

candidate_df = pd.DataFrame(candidate_rows).sort_values("test_accuracy", ascending=False)
print("\nTest Set comparison:")
print(candidate_df.to_string(index=False))

chosen_model_name = candidate_df.iloc[0]["model"]
chosen_model = candidate_models[chosen_model_name]
print(f"\nChosen final model: {chosen_model_name}")